# Transferencia secuencial desde imágenes de laboratorio para clasificar enfermedades foliares de la papa en campo

**Grupo 3 · Redes Neuronales 2026-II · Escuela Profesional de Ingeniería de Sistemas · Universidad Nacional Tecnológica de Lima Sur**

Integrantes: Jaqueline María Álvarez Rocca (2223010293) · Diego Armando Gamero Palacios (2223010229) ·
Carlos Alberto de Jesús Meza Peláez (2223110229) · Carlos Steven Santiago Flores (2223110056).
Docente: Mg. Javier A. Azabache Asmat.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StevenSntg/papa-transferencia-secuencial/blob/main/Modelo_Investigacion.ipynb)

Este cuaderno entrena y evalúa los cuatro experimentos de la Sección II del artículo con una misma red,
**EfficientNetV2B0**, y tres estrategias de inicialización:

| Estrategia | Pesos iniciales | Papel |
|---|---|---|
| S0 · desde cero | aleatorios | cota inferior |
| S1 · transferencia directa | ImageNet | línea base |
| S2 · transferencia secuencial | ImageNet, luego PlantVillage (3 clases) | propuesta |

| Experimento | Pregunta | Qué se hace |
|---|---|---|
| E1 | PI1 · brecha laboratorio–campo | El modelo ajustado con PlantVillage se evalúa, sin reajuste, en las 1248 fotos de campo de las tres clases comparables |
| E2 | PI2 · ¿mejora la transferencia secuencial? | S0, S1 y S2 con la partición depurada (Tabla II): F1 macro, sensibilidad por clase, IC 95 % y McNemar |
| E3 | PI3 · efecto de la partición | S1 con una partición aleatoria de las 3076 fotos originales, frente a la depurada |
| E4 | PI4 · ¿lesión o fondo? | Matrices de confusión y mapas Grad-CAM de aciertos y errores |

**Cómo ejecutarlo en Google Colab**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución →* **GPU T4**.
2. *Entorno de ejecución →* **Ejecutar todas**.
3. Autorizar el acceso a Google Drive cuando se solicite: allí se guardan la caché de imágenes, los modelos y los resultados (carpeta `MyDrive/TF_Grupo3_RN/`).

**Reanudación.** Cada corrida terminada queda guardada en Drive. Si Colab se desconecta, basta con volver a *Ejecutar todas*: las corridas hechas se saltan y solo se repite la que estaba a medias.

**Requisitos.** GPU (T4 de Colab), unos 6 GB de RAM y 2 GB libres en Drive. Tiempo estimado de la corrida completa con GPU T4: entre 1,5 y 3 horas (el tiempo real queda en `resultados/entorno.json`). No se necesita ninguna credencial: los conjuntos de datos se descargan de fuentes públicas.

**La sección 13 (DEMO)** prueba el modelo con cinco fotografías sin reentrenar.

## 1. Entorno y dependencias

In [ ]:
# 1.1 · Código del repositorio. En Colab se clona; en local se ejecuta desde la carpeta 02_CODIGO.
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPOSITORIO = "https://github.com/StevenSntg/papa-transferencia-secuencial.git"
if EN_COLAB:
    destino_repo = Path("/content/papa-transferencia-secuencial")
    if destino_repo.is_dir():
        subprocess.run(["git", "-C", str(destino_repo), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPOSITORIO, str(destino_repo)], check=True)
    os.chdir(destino_repo)
RAIZ = Path.cwd()
print("Carpeta del código:", RAIZ)

In [ ]:
# 1.2 · Instalación de dependencias (requirements.txt). En Colab casi todas vienen preinstaladas;
# las versiones exactas con que se obtuvieron los resultados quedan en resultados/entorno.json.
if EN_COLAB:
    %pip install -q -r requirements.txt

In [ ]:
# 1.3 · Librerías
import gc
import json
import platform
import shutil
import time
import traceback
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import scipy
import sklearn
import tensorflow as tf
import keras
from keras import layers
from PIL import Image, ImageOps
from scipy.stats import binomtest
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

tf.get_logger().setLevel("ERROR")      # oculta avisos internos de TensorFlow (p. ej., retrazado de funciones)
GPU = tf.config.list_physical_devices("GPU")


def nombre_gpu():
    try:
        salida = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                                capture_output=True, text=True, timeout=20)
        return salida.stdout.strip() or "sin GPU"
    except Exception:
        return "sin GPU"


ENTORNO = {
    "fecha_inicio": datetime.now().isoformat(timespec="seconds"),
    "entorno": "Google Colab" if EN_COLAB else "local",
    "sistema": platform.platform(),
    "python": platform.python_version(),
    "gpu": nombre_gpu(),
    "versiones": {
        "tensorflow": tf.__version__, "keras": keras.__version__, "numpy": np.__version__,
        "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "scipy": scipy.__version__,
        "matplotlib": matplotlib.__version__, "pillow": PIL.__version__,
    },
}
try:
    import psutil
    ENTORNO["ram_gb"] = round(psutil.virtual_memory().total / 2**30, 1)
except ImportError:
    pass
print(json.dumps(ENTORNO, indent=1, ensure_ascii=False))
if EN_COLAB and not GPU:
    print("AVISO: no hay GPU. Cambie el tipo de entorno de ejecución a GPU T4 antes de entrenar.")

## 2. Configuración

Los valores son los de la Tabla III del artículo. `MODO = "prueba"` ejecuta todo el cuaderno con unas
pocas fotos y una época por fase, solo para comprobar que el código funciona (no produce resultados válidos).

In [ ]:
MODO = os.environ.get("MODO_CUADERNO", "completo")   # "completo" | "prueba"
assert MODO in ("completo", "prueba"), MODO
PRUEBA = MODO == "prueba"

# Carpetas de trabajo: en Colab, Google Drive (persiste entre sesiones); en local, junto al código.
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    TRABAJO = Path("/content/drive/MyDrive/TF_Grupo3_RN" + ("_prueba" if PRUEBA else ""))
    DATOS = Path("/content/datos")
else:
    TRABAJO = Path(os.environ.get("TRABAJO_CUADERNO", RAIZ / ("_prueba" if PRUEBA else "_local")))
    DATOS = Path(os.environ.get("DATOS_TF", RAIZ.parent / "datos"))
CACHE = TRABAJO / "cache"
MODELOS = TRABAJO / "modelos"
RESULTADOS = TRABAJO / "resultados"
CORRIDAS_DIR = RESULTADOS / "corridas"
FIGURAS = RESULTADOS / "figuras"
for carpeta in (CACHE, MODELOS, CORRIDAS_DIR, FIGURAS):
    carpeta.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "tam_imagen": 224,        # 224 × 224 × 3, interpolación bilineal
    "lote": 32,
    "dropout": 0.2,
    "lr_cabeza": 1e-3,        # fase 1 de la transferencia: solo la capa de salida
    "epocas_cabeza": 5,
    "lr_ajuste": 1e-4,        # fase 2: ajuste fino de todas las capas
    "epocas_ajuste": 50,
    "lr_cero": 1e-3,          # S0: entrenamiento desde pesos aleatorios
    "epocas_cero": 100,
    "paciencia": 10,          # parada temprana sobre la pérdida de validación
    "semillas": [1, 2, 3],
    "semilla_particion": 42,  # la misma de las particiones de la Tabla II
    "remuestreos": 2000,      # réplicas para los intervalos de confianza del 95 %
    "precision_mixta": bool(GPU),
}
if PRUEBA:
    CONFIG.update(epocas_cabeza=1, epocas_ajuste=1, epocas_cero=1, paciencia=1, semillas=[1], remuestreos=50)
TAM = CONFIG["tam_imagen"]

# Precisión mixta en GPU (float16 en las capas internas y float32 en la salida): reduce el tiempo de
# entrenamiento a casi la mitad en la T4. En CPU se usa float32.
keras.mixed_precision.set_global_policy("mixed_float16" if CONFIG["precision_mixta"] else "float32")

# Clases. Las carpetas del conjunto de campo se llaman así en Mendeley Data («Phytopthora», sic).
CLASES_CAMPO = ["Bacteria", "Fungi", "Healthy", "Nematode", "Pest", "Phytopthora", "Virus"]
CLASES_LAB = ["Potato___Early_blight", "Potato___Late_blight", "Potato___healthy"]
NOMBRE = {"Bacteria": "Bacterias", "Fungi": "Hongos", "Healthy": "Sana", "Nematode": "Nematodos", "Pest": "Plagas",
          "Phytopthora": "Phytophthora", "Virus": "Virus", "Potato___Early_blight": "Tizón temprano",
          "Potato___Late_blight": "Tizón tardío", "Potato___healthy": "Sana (lab.)"}
# Correspondencia de E1 entre campo y laboratorio (la de hongos con tizón temprano es aproximada).
CAMPO_A_LAB = {"Fungi": "Potato___Early_blight", "Phytopthora": "Potato___Late_blight", "Healthy": "Potato___healthy"}
PARTICIONES = ["entrenamiento", "validacion", "prueba"]

print("Modo:", MODO, "· trabajo en", TRABAJO)
print(json.dumps(CONFIG, indent=1))

## 3. Datos

* **Campo:** Shabrina *et al.* (2024), *Potato Leaf Disease Dataset in Uncontrolled Environment*, Mendeley Data,
  DOI 10.17632/ptz377bwb8.1, licencia CC BY 4.0. 3076 JPG de 1500 × 1500 px, siete clases.
* **Laboratorio:** subconjunto de papa de PlantVillage (Hughes y Salathé), repositorio de Mohanty *et al.*,
  licencia CC BY-SA 3.0. 2152 JPG de 256 × 256 px, tres clases.

Las particiones de `particiones/` provienen de `scripts/auditar_y_particionar.py`: se retiraron 49 copias
exactas y 12 fotos con etiqueta contradictoria (quedan 3015) y los 14 pares de casi duplicados van juntos a una
misma partición. Las imágenes se redimensionan una sola vez a 224 × 224 y se guardan como matrices `uint8`
en la caché de Drive. **Los píxeles se dejan en 0–255**: EfficientNetV2B0 incluye sus propias capas de
reescalado y normalización con la media y la desviación de ImageNet.

In [ ]:
part_campo = pd.read_csv(RAIZ / "particiones" / "campo_particion.csv")
part_lab = pd.read_csv(RAIZ / "particiones" / "laboratorio_particion.csv")


def submuestra(df, n_por_particion, n_descartadas=0):
    """Solo para MODO = "prueba": unas pocas fotos de cada clase y partición."""
    partes = [g.head(n_por_particion[p]) for (_, p), g in
              df[df["estado"] == "conservada"].groupby(["clase", "particion"])]
    partes.append(df[df["estado"] != "conservada"].head(n_descartadas))
    return pd.concat(partes).sort_index().reset_index(drop=True)


if PRUEBA:
    part_campo = submuestra(part_campo, {"entrenamiento": 6, "validacion": 2, "prueba": 2}, n_descartadas=4)
    part_lab = submuestra(part_lab, {"entrenamiento": 6, "validacion": 2, "prueba": 2})

print("Campo:", len(part_campo), "filas ·", part_campo["estado"].value_counts().to_dict())
print("Laboratorio:", len(part_lab), "filas")

In [ ]:
CARPETA_CONJUNTO = {"campo": "campo_shabrina", "laboratorio": "plantvillage_papa"}


def ruta_imagen(conjunto, clase, archivo):
    return DATOS / CARPETA_CONJUNTO[conjunto] / clase / archivo


def leer_224(ruta):
    """Abre una foto, la pasa a RGB y la redimensiona a 224 × 224 con interpolación bilineal (uint8, 0–255)."""
    with Image.open(ruta) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        return np.asarray(im.resize((TAM, TAM), Image.BILINEAR), dtype=np.uint8)


def asegurar_descarga(rutas):
    faltan = [r for r in rutas if not r.exists()]
    if faltan:
        print(f"Faltan {len(faltan)} imágenes en {DATOS}: se descargan los conjuntos (unos 770 MB)…")
        subprocess.run([sys.executable, str(RAIZ / "scripts" / "descargar_datasets.py"), "--destino", str(DATOS)],
                       check=False)
        faltan = [r for r in rutas if not r.exists()]
        if faltan:
            raise FileNotFoundError(f"Siguen faltando {len(faltan)} imágenes, p. ej. {faltan[0]}")


def cargar_cache(conjunto, df):
    """Devuelve las imágenes del CSV de partición, en su mismo orden, como matriz N × 224 × 224 × 3."""
    npy, indice = CACHE / f"{conjunto}_{TAM}.npy", CACHE / f"{conjunto}_indice.csv"
    if npy.exists() and indice.exists():
        previo = pd.read_csv(indice)
        if previo["clase"].tolist() == df["clase"].tolist() and previo["archivo"].tolist() == df["archivo"].tolist():
            print(f"{conjunto}: caché encontrada ({npy.stat().st_size / 1e6:.0f} MB)")
            return np.load(npy)
    rutas = [ruta_imagen(conjunto, c, a) for c, a in zip(df["clase"], df["archivo"])]
    asegurar_descarga(rutas)
    t = time.time()
    with ThreadPoolExecutor(max_workers=8) as ex:
        X = np.stack(list(ex.map(leer_224, rutas)))
    np.save(npy, X)
    df[["clase", "archivo", "sha256"]].to_csv(indice, index=False)
    print(f"{conjunto}: {len(X)} imágenes redimensionadas en {time.time() - t:.0f} s")
    return X


X_campo = cargar_cache("campo", part_campo)
X_lab = cargar_cache("laboratorio", part_lab)
y_campo = part_campo["clase"].map({c: i for i, c in enumerate(CLASES_CAMPO)}).to_numpy()
y_lab = part_lab["clase"].map({c: i for i, c in enumerate(CLASES_LAB)}).to_numpy()
print("X_campo", X_campo.shape, X_campo.dtype, "· X_lab", X_lab.shape)

In [ ]:
# Índices de cada partición. Depurada (Tabla II): solo las fotos conservadas.
conservada = (part_campo["estado"] == "conservada").to_numpy()
IDX_DEP = {p: np.where(conservada & (part_campo["particion"] == p).to_numpy())[0] for p in PARTICIONES}
IDX_LAB = {p: np.where((part_lab["particion"] == p).to_numpy())[0] for p in PARTICIONES}

# E1: todas las fotos conservadas de campo de las tres clases comparables (1248 en modo completo).
IDX_CAMPO3 = np.where(conservada & part_campo["clase"].isin(list(CAMPO_A_LAB)).to_numpy())[0]

# E3: partición aleatoria estratificada 70/15/15 de TODAS las fotos originales (sin depurar), como en los
# estudios revisados; la semilla es la misma de la partición depurada.
todas = np.arange(len(part_campo))
i_ent, i_resto = train_test_split(todas, test_size=0.30, stratify=y_campo, random_state=CONFIG["semilla_particion"])
i_val, i_pru = train_test_split(i_resto, test_size=0.50, stratify=y_campo[i_resto],
                                random_state=CONFIG["semilla_particion"])
IDX_E3 = {"entrenamiento": np.sort(i_ent), "validacion": np.sort(i_val), "prueba": np.sort(i_pru)}
particion_e3 = part_campo[["clase", "archivo", "sha256", "estado"]].copy()
particion_e3["particion"] = ""
for p, ind in IDX_E3.items():
    particion_e3.loc[ind, "particion"] = p
particion_e3.to_csv(RESULTADOS / "e3_particion_aleatoria.csv", index=False)


def tabla_conteos(df, idx, clases):
    filas = {NOMBRE[c]: [int((df["clase"].iloc[idx[p]] == c).sum()) for p in PARTICIONES] for c in clases}
    t = pd.DataFrame(filas, index=["Entrenamiento", "Validación", "Prueba"]).T
    t.loc["Total"] = t.sum()
    return t


print("Partición depurada de campo (Tabla II):")
display(tabla_conteos(part_campo, IDX_DEP, CLASES_CAMPO))
print("Partición de laboratorio:")
display(tabla_conteos(part_lab, IDX_LAB, CLASES_LAB))
print("Partición aleatoria de E3 (sin depurar):")
display(tabla_conteos(part_campo, IDX_E3, CLASES_CAMPO))
print("Fotos de campo de las tres clases comparables (E1):", len(IDX_CAMPO3))

In [ ]:
# Una foto de muestra por clase de cada conjunto.
fig, ejes = plt.subplots(2, 7, figsize=(17, 5.4))
for j, c in enumerate(CLASES_CAMPO):
    i = IDX_DEP["entrenamiento"][y_campo[IDX_DEP["entrenamiento"]] == j][0]
    ejes[0, j].imshow(X_campo[i]); ejes[0, j].set_title(f"Campo · {NOMBRE[c]}", fontsize=9)
for j, c in enumerate(CLASES_LAB):
    i = IDX_LAB["entrenamiento"][y_lab[IDX_LAB["entrenamiento"]] == j][0]
    ejes[1, j].imshow(X_lab[i]); ejes[1, j].set_title(f"Laboratorio · {NOMBRE[c]}", fontsize=9)
for ax in ejes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.savefig(FIGURAS / "muestras_conjuntos.png", dpi=150); plt.show()

## 4. Preprocesamiento y aumento de datos

Solo el conjunto de entrenamiento se aumenta: volteo horizontal y vertical, rotación de hasta 20°,
ampliación de hasta 10 % y variaciones de brillo y de contraste de hasta 10 %. El fondo no se segmenta.
El aumento se aplica dentro del grafo de entrenamiento (en la GPU) y no forma parte del modelo guardado.

In [ ]:
def capa_aumento(semilla):
    return keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical", seed=semilla, dtype="float32"),
        layers.RandomRotation(20 / 360, fill_mode="reflect", seed=semilla, dtype="float32"),
        layers.RandomZoom((-0.10, 0.0), fill_mode="reflect", seed=semilla, dtype="float32"),
        layers.RandomBrightness(0.10, value_range=(0, 255), seed=semilla, dtype="float32"),
        layers.RandomContrast(0.10, seed=semilla, dtype="float32"),
    ], name="aumento")


def pesos_por_clase(y, k):
    """Pesos inversos a la frecuencia: w_c = N / (K · n_c)."""
    n = np.bincount(y, minlength=k).astype("float64")
    return (len(y) / (k * np.maximum(n, 1))).astype("float32")


def crear_ds(X, y, k, pesos=None, barajar=False, semilla=0):
    """tf.data con lotes de 32; si se pasan pesos por clase, cada foto lleva el peso de su clase."""
    Y = keras.utils.to_categorical(y, k).astype("float32")
    tensores = (X, Y) if pesos is None else (X, Y, pesos[y])
    ds = tf.data.Dataset.from_tensor_slices(tensores)
    if barajar:
        ds = ds.shuffle(len(y), seed=semilla, reshuffle_each_iteration=True)
    ds = ds.batch(CONFIG["lote"]).map(lambda x, *r: (tf.cast(x, tf.float32), *r), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


def crear_ds_prediccion(X):
    return (tf.data.Dataset.from_tensor_slices(X).batch(64)
            .map(lambda x: tf.cast(x, tf.float32)).prefetch(tf.data.AUTOTUNE))


# Ejemplo: ocho versiones aumentadas de una misma foto de entrenamiento.
aum = capa_aumento(0)
foto = tf.cast(X_campo[IDX_DEP["entrenamiento"][0]][None], tf.float32)
fig, ejes = plt.subplots(1, 8, figsize=(17, 2.6))
for ax in ejes:
    ax.imshow(np.clip(aum(foto, training=True)[0].numpy(), 0, 255).astype("uint8")); ax.axis("off")
plt.suptitle("Aumento de datos aplicado a una foto de entrenamiento"); plt.tight_layout()
plt.savefig(FIGURAS / "aumento_ejemplo.png", dpi=120); plt.show()
del aum

## 5. Modelo

EfficientNetV2B0 (270 capas, 5 919 312 parámetros en el extractor, activación SiLU) + promedio global +
*dropout* 0,2 + capa densa *softmax* (7 neuronas en campo, 3 en laboratorio).

In [ ]:
CAPAS_CABEZA = {"promedio_global", "dropout"}


def construir_modelo(n_clases, pesos):
    """pesos = "imagenet" (S1, S2 y PlantVillage) o None (S0, pesos aleatorios)."""
    entrada = keras.Input((TAM, TAM, 3), name="imagen")
    base = keras.applications.EfficientNetV2B0(include_top=False, weights=pesos, input_tensor=entrada)
    x = layers.GlobalAveragePooling2D(name="promedio_global")(base.output)
    x = layers.Dropout(CONFIG["dropout"], name="dropout")(x)
    salida = layers.Dense(n_clases, activation="softmax", dtype="float32", name=f"salida_{n_clases}")(x)
    return keras.Model(entrada, salida, name=f"efficientnetv2b0_{n_clases}_clases")


def cambiar_cabeza(modelo, n_clases):
    """S2: conserva todo hasta el dropout y pone una capa de salida nueva (pesos aleatorios)."""
    x = modelo.get_layer("dropout").output
    salida = layers.Dense(n_clases, activation="softmax", dtype="float32", name=f"salida_{n_clases}")(x)
    return keras.Model(modelo.input, salida, name=f"efficientnetv2b0_{n_clases}_clases_secuencial")


def poner_entrenables(modelo, modo):
    """cabeza: solo la salida · ajuste: todo salvo la normalización por lotes (queda en modo inferencia,
    como recomienda la guía de Keras para el ajuste fino) · cero: todas las capas."""
    for capa in modelo.layers:
        es_cabeza = capa.name in CAPAS_CABEZA or capa.name.startswith("salida_")
        if modo == "cabeza":
            capa.trainable = es_cabeza
        elif modo == "ajuste":
            capa.trainable = es_cabeza or not isinstance(capa, layers.BatchNormalization)
        else:
            capa.trainable = True


def contar(pesos):
    return int(sum(np.prod(v.shape) for v in pesos))


m = construir_modelo(7, "imagenet")
n_extractor = contar(m.weights) - contar(m.get_layer("salida_7").weights)
print(f"Capas del modelo: {len(m.layers)} · parámetros del extractor: {n_extractor:,} · "
      f"de la cabeza: {contar(m.get_layer('salida_7').weights):,}".replace(",", " "))
for modo in ("cabeza", "ajuste", "cero"):
    poner_entrenables(m, modo)
    print(f"  modo {modo:<7}: {contar(m.trainable_weights):>10,} parámetros entrenables".replace(",", " "))
del m
keras.backend.clear_session()

## 6. Entrenamiento

Quince corridas, en orden de importancia: PlantVillage (necesaria para E1 y S2), S1, S2, S0 y E3, con tres
semillas cada una. Cada corrida guarda en Drive su modelo (`modelos/<corrida>.keras`), su historial y las
probabilidades predichas para validación y prueba (`resultados/corridas/<corrida>/`). Una corrida con
`meta.json` se considera terminada y se salta. La pérdida de validación usa los mismos pesos por clase que
el entrenamiento; la parada temprana restaura los pesos de la época con menor pérdida de validación.

In [ ]:
class MejoresPesos(keras.callbacks.Callback):
    """Guarda los pesos de la época con menor val_loss y los restaura al terminar."""

    def __init__(self):
        super().__init__()
        self.mejor, self.epoca, self.pesos = np.inf, None, None

    def on_epoch_end(self, epoch, logs=None):
        v = (logs or {}).get("val_loss")
        if v is not None and v < self.mejor:
            self.mejor, self.epoca, self.pesos = v, epoch + 1, self.model.get_weights()

    def on_train_end(self, logs=None):
        if self.pesos is not None:
            self.model.set_weights(self.pesos)


def ajustar(modelo, datos, lr, epocas, semilla, fase, seleccionar):
    """Una fase de entrenamiento. El aumento va en una envoltura que comparte los pesos con el modelo."""
    entrada = keras.Input((TAM, TAM, 3), name="entrada")
    envoltura = keras.Model(entrada, modelo(capa_aumento(semilla)(entrada)), name="entrenamiento")
    envoltura.compile(optimizer=keras.optimizers.Adam(lr), loss="categorical_crossentropy",
                      metrics=["accuracy"], jit_compile=False)
    callbacks, mejores = [], None
    if seleccionar:
        mejores = MejoresPesos()
        callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=CONFIG["paciencia"]), mejores]
    print(f"  fase {fase}: lr={lr:g}, hasta {epocas} épocas, "
          f"{contar(modelo.trainable_weights):,} parámetros entrenables".replace(",", " "))
    h = envoltura.fit(datos["ent"], validation_data=datos["val"], epochs=epocas, callbacks=callbacks, verbose=2)
    hist = pd.DataFrame(h.history)
    hist.insert(0, "epoca", np.arange(1, len(hist) + 1))
    hist.insert(0, "fase", fase)
    return hist, (mejores.epoca if mejores else len(hist))


def entrenar(modelo, datos, semilla, desde_cero):
    if desde_cero:
        poner_entrenables(modelo, "cero")
        h, mejor = ajustar(modelo, datos, CONFIG["lr_cero"], CONFIG["epocas_cero"], semilla, "cero", True)
        return h, {"epocas_cero": len(h), "mejor_epoca": mejor}
    poner_entrenables(modelo, "cabeza")
    h1, _ = ajustar(modelo, datos, CONFIG["lr_cabeza"], CONFIG["epocas_cabeza"], semilla, "cabeza", False)
    poner_entrenables(modelo, "ajuste")
    h2, mejor = ajustar(modelo, datos, CONFIG["lr_ajuste"], CONFIG["epocas_ajuste"], semilla, "ajuste", True)
    return pd.concat([h1, h2], ignore_index=True), {"epocas_cabeza": len(h1), "epocas_ajuste": len(h2),
                                                   "mejor_epoca_ajuste": mejor}


# Qué datos usa cada estrategia: (matriz, etiquetas, filas, índices, clases, conjuntos a predecir)
def especificacion(estrategia):
    if estrategia == "pv":
        return X_lab, y_lab, part_lab, IDX_LAB, CLASES_LAB, {"validacion": IDX_LAB["validacion"],
                                                             "prueba": IDX_LAB["prueba"]}
    idx = IDX_E3 if estrategia == "e3" else IDX_DEP
    return X_campo, y_campo, part_campo, idx, CLASES_CAMPO, {"validacion": idx["validacion"], "prueba": idx["prueba"]}


def guardar_probabilidades(modelo, X, filas, clases, ruta):
    p = modelo.predict(crear_ds_prediccion(X), verbose=0).astype("float64")
    tabla = filas[["clase", "archivo"]].reset_index(drop=True).copy()
    for j, c in enumerate(clases):
        tabla[f"p_{c}"] = p[:, j].round(6)
    tabla["prediccion"] = [clases[k] for k in p.argmax(axis=1)]
    tabla.to_csv(ruta, index=False)
    return tabla


def f1_macro(y, p, k):
    """F1 macro a partir de la matriz de confusión (las clases sin muestras ni predicciones cuentan 0)."""
    cm = np.bincount(np.asarray(y) * k + np.asarray(p), minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    den = 2 * tp + (cm.sum(0) - tp) + (cm.sum(1) - tp)
    return float(np.mean(np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)))


def ejecutar_corrida(estrategia, semilla):
    nombre = f"{estrategia}_s{semilla}"
    carpeta = CORRIDAS_DIR / nombre
    if (carpeta / "meta.json").exists():
        print(f"{nombre}: ya estaba hecha, se salta")
        return
    carpeta.mkdir(parents=True, exist_ok=True)
    keras.backend.clear_session(); gc.collect()
    keras.utils.set_random_seed(semilla)
    X, y, filas, idx, clases, a_predecir = especificacion(estrategia)
    k = len(clases)
    pesos = pesos_por_clase(y[idx["entrenamiento"]], k)
    datos = {"ent": crear_ds(X[idx["entrenamiento"]], y[idx["entrenamiento"]], k, pesos, True, semilla),
             "val": crear_ds(X[idx["validacion"]], y[idx["validacion"]], k, pesos)}
    if estrategia == "s2":
        previo = MODELOS / f"pv_s{semilla}.keras"
        if not previo.exists():
            raise RuntimeError(f"S2 necesita antes la corrida PlantVillage {previo.name}")
        modelo = cambiar_cabeza(keras.models.load_model(previo), k)
    else:
        modelo = construir_modelo(k, None if estrategia == "s0" else "imagenet")
    print(f"\n=== {nombre} · {len(idx['entrenamiento'])} fotos de entrenamiento · {k} clases ===")
    t = time.time()
    hist, epocas = entrenar(modelo, datos, semilla, desde_cero=(estrategia == "s0"))
    segundos = time.time() - t
    hist.to_csv(carpeta / "historial.csv", index=False)
    modelo.save(MODELOS / f"{nombre}.keras")          # sin optimizador: el modelo interno no se compila
    meta = {"corrida": nombre, "estrategia": estrategia, "semilla": semilla, "clases": clases,
            "segundos_entrenamiento": round(segundos, 1), **epocas,
            "pesos_por_clase": dict(zip(clases, pesos.round(4).tolist()))}
    for conjunto, ind in a_predecir.items():
        tabla = guardar_probabilidades(modelo, X[ind], filas.iloc[ind], clases, carpeta / f"prob_{conjunto}.csv")
        yr = tabla["clase"].map({c: i for i, c in enumerate(clases)}).to_numpy()
        yp = tabla["prediccion"].map({c: i for i, c in enumerate(clases)}).to_numpy()
        meta[f"exactitud_{conjunto}"] = round(float((yr == yp).mean()), 4)
        meta[f"f1_macro_{conjunto}"] = round(f1_macro(yr, yp, k), 4)
    if estrategia == "pv":                              # E1: las 1248 fotos de campo comparables
        guardar_probabilidades(modelo, X_campo[IDX_CAMPO3], part_campo.iloc[IDX_CAMPO3], CLASES_LAB,
                               carpeta / "prob_campo3.csv")
    meta["fecha_fin"] = datetime.now().isoformat(timespec="seconds")
    (carpeta / "meta.json").write_text(json.dumps(meta, indent=1, ensure_ascii=False), encoding="utf-8")
    print(f"{nombre}: {segundos / 60:.1f} min · exactitud val {meta['exactitud_validacion']:.3f} · "
          f"F1 macro val {meta['f1_macro_validacion']:.3f}")
    del modelo, datos


ESTRATEGIAS = ["pv", "s1", "s2", "s0", "e3"]
CORRIDAS = [(e, s) for e in ESTRATEGIAS for s in CONFIG["semillas"]]
pendientes = [f"{e}_s{s}" for e, s in CORRIDAS if not (CORRIDAS_DIR / f"{e}_s{s}" / "meta.json").exists()]
print(f"Corridas: {len(CORRIDAS)} · pendientes: {len(pendientes)} {pendientes}")

In [ ]:
# Bucle de entrenamiento. Una corrida que falla queda registrada y no detiene las siguientes.
t_total, fallidas = time.time(), []
for estrategia, semilla in CORRIDAS:
    try:
        ejecutar_corrida(estrategia, semilla)
    except Exception:
        traceback.print_exc()
        fallidas.append(f"{estrategia}_s{semilla}")
ENTORNO["minutos_entrenamiento_sesion"] = round((time.time() - t_total) / 60, 1)
print(f"\nEntrenamiento de esta sesión: {ENTORNO['minutos_entrenamiento_sesion']} min · fallidas: {fallidas or 'ninguna'}")

## 7–9. Evaluación: E1, E2 y E3

Todo lo que sigue se calcula a partir de las probabilidades guardadas en `resultados/corridas/`, de modo que
puede repetirse sin GPU y sin los modelos. Los intervalos de confianza del 95 % se obtienen por remuestreo
percentil de las fotos de prueba (2000 réplicas); en cada réplica se promedia la métrica de las tres semillas.

In [ ]:
def leer_corrida(nombre):
    carpeta = CORRIDAS_DIR / nombre
    if not (carpeta / "meta.json").exists():
        return None
    out = {"meta": json.loads((carpeta / "meta.json").read_text(encoding="utf-8"))}
    for f in carpeta.glob("prob_*.csv"):
        out[f.stem.removeprefix("prob_")] = pd.read_csv(f)
    return out


def etiquetas(tabla, clases, columna="clase", mapa=None):
    serie = tabla[columna].map(mapa) if mapa else tabla[columna]
    return serie.map({c: i for i, c in enumerate(clases)}).to_numpy()


def corridas_de(estrategia):
    return {s: r for s in CONFIG["semillas"] if (r := leer_corrida(f"{estrategia}_s{s}")) is not None}


def media_de(valores):
    v = np.asarray(valores, dtype=float)
    return {"media": round(float(v.mean()), 4), "de": round(float(v.std(ddof=1)) if len(v) > 1 else 0.0, 4),
            "por_semilla": [round(float(x), 4) for x in v]}


def remuestreo(n, B, semilla=2026):
    rng = np.random.default_rng(semilla)
    return (rng.integers(0, n, n) for _ in range(B))


def ic95(valores):
    lo, hi = np.percentile(valores, [2.5, 97.5])
    return [round(float(lo), 4), round(float(hi), 4)]


def metricas_por_clase(y, p, clases):
    prec, sens, f1, sop = precision_recall_fscore_support(y, p, labels=range(len(clases)), zero_division=0)
    return pd.DataFrame({"precision": prec, "sensibilidad": sens, "f1": f1, "soporte": sop},
                        index=[NOMBRE[c] for c in clases])


RESUMEN = {"modo": MODO, "pendientes": [], "config": CONFIG}

In [ ]:
# ---- E1 · brecha laboratorio–campo (PI1)
pv = corridas_de("pv")
if pv:
    acc_lab, acc_campo, brechas, sens_lab, sens_campo, mc_campo = [], [], [], [], [], 0
    for s, r in pv.items():
        yl, pl = etiquetas(r["prueba"], CLASES_LAB), etiquetas(r["prueba"], CLASES_LAB, "prediccion")
        yc = etiquetas(r["campo3"], CLASES_LAB, mapa=CAMPO_A_LAB)
        pc = etiquetas(r["campo3"], CLASES_LAB, "prediccion")
        acc_lab.append((yl == pl).mean()); acc_campo.append((yc == pc).mean())
        brechas.append(100 * (acc_lab[-1] - acc_campo[-1]))
        sens_lab.append(metricas_por_clase(yl, pl, CLASES_LAB)["sensibilidad"])
        sens_campo.append(metricas_por_clase(yc, pc, CLASES_LAB)["sensibilidad"])
        mc_campo = mc_campo + confusion_matrix(yc, pc, labels=range(3))
    # IC de la brecha: se remuestrean por separado la prueba de laboratorio y las fotos de campo.
    ref = next(iter(pv.values()))
    nl, nc = len(ref["prueba"]), len(ref["campo3"])
    acierto_l = [etiquetas(r["prueba"], CLASES_LAB) == etiquetas(r["prueba"], CLASES_LAB, "prediccion") for r in pv.values()]
    acierto_c = [etiquetas(r["campo3"], CLASES_LAB, mapa=CAMPO_A_LAB) == etiquetas(r["campo3"], CLASES_LAB, "prediccion")
                 for r in pv.values()]
    rb = [100 * np.mean([a[il].mean() - c[ic].mean() for a, c in zip(acierto_l, acierto_c)])
          for il, ic in zip(remuestreo(nl, CONFIG["remuestreos"], 1), remuestreo(nc, CONFIG["remuestreos"], 2))]
    sens_l, sens_c = pd.concat(sens_lab, axis=1).mean(axis=1), pd.concat(sens_campo, axis=1).mean(axis=1)
    e1_clases = pd.DataFrame({"Sensibilidad laboratorio": sens_l, "Sensibilidad campo": sens_c,
                              "Diferencia (pp)": 100 * (sens_l - sens_c)})
    RESUMEN["e1"] = {
        "n_prueba_laboratorio": nl, "n_campo_comparables": nc,
        "exactitud_laboratorio": media_de(acc_lab), "exactitud_campo": media_de(acc_campo),
        "brecha_pp": {**media_de(brechas), "ic95": ic95(rb)},
        "por_clase": {i: {k: round(float(v), 4) for k, v in fila.items()} for i, fila in e1_clases.iterrows()},
        "matriz_confusion_campo_suma_semillas": {"clases": CLASES_LAB, "filas_reales": mc_campo.tolist()},
    }
    e1_clases.round(4).to_csv(RESULTADOS / "e1_por_clase.csv")
    print(f"E1 · exactitud en laboratorio {np.mean(acc_lab):.3f} · en campo {np.mean(acc_campo):.3f} · "
          f"brecha {np.mean(brechas):.1f} pp (IC 95 % {ic95(rb)[0]:.1f}–{ic95(rb)[1]:.1f})")
    display(e1_clases.round(3))
else:
    RESUMEN["pendientes"].append("E1 (faltan las corridas PlantVillage)")

In [ ]:
# ---- E2 · S0, S1 y S2 con la partición depurada (PI2)
NOMBRE_ESTRATEGIA = {"s0": "S0 · desde cero", "s1": "S1 · transferencia directa", "s2": "S2 · transferencia secuencial"}
K = len(CLASES_CAMPO)
# Las probabilidades de prueba de S0, S1 y S2 están en el orden de IDX_DEP["prueba"].
y_ref = etiquetas(part_campo.iloc[IDX_DEP["prueba"]], CLASES_CAMPO)
e2 = {e: corridas_de(e) for e in ("s0", "s1", "s2")}
e2 = {e: r for e, r in e2.items() if r}
pred_prueba, filas_e2, por_clase_e2, mc_e2 = {}, {}, {}, {}
for e, rs in e2.items():
    pred_prueba[e] = {s: etiquetas(r["prueba"], CLASES_CAMPO, "prediccion") for s, r in rs.items()}
    f1s = [f1_macro(y_ref, p, K) for p in pred_prueba[e].values()]
    accs = [(y_ref == p).mean() for p in pred_prueba[e].values()]
    boot = [np.mean([f1_macro(y_ref[i], p[i], K) for p in pred_prueba[e].values()])
            for i in remuestreo(len(y_ref), CONFIG["remuestreos"])]
    pc = pd.concat([metricas_por_clase(y_ref, p, CLASES_CAMPO) for p in pred_prueba[e].values()]).groupby(level=0).mean()
    por_clase_e2[e] = pc.reindex([NOMBRE[c] for c in CLASES_CAMPO])
    mc_e2[e] = sum(confusion_matrix(y_ref, p, labels=range(K)) for p in pred_prueba[e].values())
    f1_val = [r["meta"]["f1_macro_validacion"] for r in rs.values()]
    filas_e2[e] = {"estrategia": NOMBRE_ESTRATEGIA[e], "semillas": len(rs),
                   "f1_macro": media_de(f1s), "f1_macro_ic95": ic95(boot), "exactitud": media_de(accs),
                   "f1_macro_validacion": media_de(f1_val),
                   "minutos_por_corrida": round(np.mean([r["meta"]["segundos_entrenamiento"] for r in rs.values()]) / 60, 1)}
    por_clase_e2[e].round(4).to_csv(RESULTADOS / f"e2_por_clase_{e}.csv")
    pd.DataFrame(mc_e2[e], index=CLASES_CAMPO, columns=CLASES_CAMPO).to_csv(RESULTADOS / f"e2_matriz_confusion_{e}.csv")

if filas_e2:
    tabla_e2 = pd.DataFrame([{
        "Estrategia": f["estrategia"], "Semillas": f["semillas"],
        "F1 macro (media ± DE)": f"{f['f1_macro']['media']:.3f} ± {f['f1_macro']['de']:.3f}",
        "IC 95 %": f"{f['f1_macro_ic95'][0]:.3f}–{f['f1_macro_ic95'][1]:.3f}",
        "Exactitud": f"{f['exactitud']['media']:.3f} ± {f['exactitud']['de']:.3f}",
        "Min/corrida": f["minutos_por_corrida"]} for f in filas_e2.values()])
    display(tabla_e2)
    sens = pd.DataFrame({NOMBRE_ESTRATEGIA[e]: pc["sensibilidad"] for e, pc in por_clase_e2.items()})
    print("Sensibilidad por clase (media de las semillas):"); display(sens.round(3))
    sens.round(4).to_csv(RESULTADOS / "e2_sensibilidad_por_clase.csv")
RESUMEN["e2"] = {"estrategias": filas_e2,
                 "por_clase": {e: pc.round(4).to_dict(orient="index") for e, pc in por_clase_e2.items()}}

In [ ]:
# McNemar exacto por pares, con la misma semilla en ambas estrategias y corrección de Holm para las tres
# comparaciones; además, la diferencia de F1 macro con su IC 95 % por remuestreo emparejado.
def holm(pvalores):
    orden = np.argsort(pvalores); m = len(pvalores); ajust = np.empty(m); acum = 0.0
    for rango, i in enumerate(orden):
        acum = max(acum, min(1.0, (m - rango) * pvalores[i])); ajust[i] = acum
    return ajust


comparaciones = []
for a, b in (("s2", "s1"), ("s1", "s0"), ("s2", "s0")):
    if a not in pred_prueba or b not in pred_prueba:
        continue
    comunes = sorted(set(pred_prueba[a]) & set(pred_prueba[b]))
    for s in comunes:
        pa, pb = pred_prueba[a][s], pred_prueba[b][s]
        solo_a = int(((pa == y_ref) & (pb != y_ref)).sum()); solo_b = int(((pa != y_ref) & (pb == y_ref)).sum())
        pval = binomtest(min(solo_a, solo_b), solo_a + solo_b, 0.5).pvalue if solo_a + solo_b else 1.0
        comparaciones.append({"par": f"{a.upper()} vs {b.upper()}", "semilla": s, "solo_acierta_primera": solo_a,
                              "solo_acierta_segunda": solo_b, "p": pval})
    boot = [np.mean([f1_macro(y_ref[i], pred_prueba[a][s][i], K) - f1_macro(y_ref[i], pred_prueba[b][s][i], K)
                     for s in comunes]) for i in remuestreo(len(y_ref), CONFIG["remuestreos"])]
    dif = np.mean([f1_macro(y_ref, pred_prueba[a][s], K) - f1_macro(y_ref, pred_prueba[b][s], K) for s in comunes])
    RESUMEN.setdefault("e2_diferencias_f1", {})[f"{a}-{b}"] = {"media": round(float(dif), 4), "ic95": ic95(boot)}
if comparaciones:
    mcn = pd.DataFrame(comparaciones).sort_values(["semilla", "par"]).reset_index(drop=True)
    mcn["p_holm"] = mcn.groupby("semilla")["p"].transform(lambda p: holm(p.to_numpy()))
    mcn.round(5).to_csv(RESULTADOS / "e2_mcnemar.csv", index=False)
    display(mcn.round(4))
    RESUMEN["e2_mcnemar"] = mcn.round(5).to_dict(orient="records")
    print("Diferencias de F1 macro (IC 95 %):", json.dumps(RESUMEN.get("e2_diferencias_f1"), ensure_ascii=False))

In [ ]:
# Matrices de confusión normalizadas por fila (suma de las semillas).
if mc_e2:
    fig, ejes = plt.subplots(1, len(mc_e2), figsize=(6.2 * len(mc_e2), 5.6))
    ejes = np.atleast_1d(ejes)
    for ax, (e, mc) in zip(ejes, mc_e2.items()):
        norm = mc / np.maximum(mc.sum(axis=1, keepdims=True), 1)
        ax.imshow(norm, cmap="Greens", vmin=0, vmax=1)
        for i in range(K):
            for j in range(K):
                ax.text(j, i, f"{norm[i, j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if norm[i, j] > 0.6 else "black")
        ax.set_xticks(range(K), [NOMBRE[c] for c in CLASES_CAMPO], rotation=45, ha="right")
        ax.set_yticks(range(K), [NOMBRE[c] for c in CLASES_CAMPO])
        ax.set_title(NOMBRE_ESTRATEGIA[e]); ax.set_xlabel("Predicha"); ax.set_ylabel("Real")
    plt.tight_layout(); plt.savefig(FIGURAS / "e2_matrices_confusion.png", dpi=150); plt.show()
else:
    RESUMEN["pendientes"].append("E2 (faltan corridas S0/S1/S2)")

In [ ]:
# ---- E3 · partición aleatoria del conjunto original frente a la depurada (PI3)
e3 = corridas_de("e3")
if e3:
    y3 = etiquetas(next(iter(e3.values()))["prueba"], CLASES_CAMPO)
    p3 = [etiquetas(r["prueba"], CLASES_CAMPO, "prediccion") for r in e3.values()]
    # Fuga de datos: fotos de prueba con una copia exacta (mismo SHA-256) o con su casi duplicado (mismo
    # grupo dHash de la auditoría) en el entrenamiento de la partición aleatoria.
    entrenamiento_e3, prueba_e3 = part_campo.iloc[IDX_E3["entrenamiento"]], part_campo.iloc[IDX_E3["prueba"]]
    con_copia = int(prueba_e3["sha256"].isin(set(entrenamiento_e3["sha256"])).sum())
    con_casi = int(prueba_e3["grupo"].dropna().isin(set(entrenamiento_e3["grupo"].dropna())).sum())
    RESUMEN["e3"] = {
        "n_prueba": int(len(y3)),
        "fotos_prueba_con_copia_exacta_en_entrenamiento": con_copia,
        "fotos_prueba_con_casi_duplicado_en_entrenamiento": con_casi,
        "fotos_prueba_descartadas_en_la_depuracion": int((prueba_e3["estado"] != "conservada").sum()),
        "aleatoria": {"f1_macro": media_de([f1_macro(y3, p, K) for p in p3]),
                      "exactitud": media_de([(y3 == p).mean() for p in p3]),
                      "f1_macro_ic95": ic95([np.mean([f1_macro(y3[i], p[i], K) for p in p3])
                                             for i in remuestreo(len(y3), CONFIG["remuestreos"])])},
    }
    if "s1" in filas_e2:
        RESUMEN["e3"]["depurada"] = {k: filas_e2["s1"][k] for k in ("f1_macro", "exactitud", "f1_macro_ic95")}
    print(json.dumps(RESUMEN["e3"], indent=1, ensure_ascii=False))
else:
    RESUMEN["pendientes"].append("E3 (faltan las corridas con partición aleatoria)")

## 10. E4 · Grad-CAM de aciertos y errores (PI4)

Para S1 y S2 se usa la semilla con mejor F1 macro de **validación** (nunca de prueba). En cada clase se
muestran el acierto más seguro y el error más seguro de la prueba de campo, con su mapa Grad-CAM sobre la
última capa convolucional (`top_activation`). Las confusiones más frecuentes salen de las matrices de E2.

In [ ]:
def mejor_semilla(estrategia):
    rs = corridas_de(estrategia)
    return max(rs, key=lambda s: rs[s]["meta"]["f1_macro_validacion"]) if rs else None


def copia_float32(modelo):
    """Copia del modelo en float32: evita que los gradientes en float16 se anulen en Grad-CAM."""
    politica = keras.mixed_precision.global_policy().name
    keras.mixed_precision.set_global_policy("float32")
    copia = construir_modelo(K, None)
    copia.set_weights(modelo.get_weights())
    keras.mixed_precision.set_global_policy(politica)
    return copia


def gradcam(modelo, imagen):
    """Mapa Grad-CAM (Selvaraju et al.) de la clase predicha, normalizado a 0–1 y llevado a 224 × 224."""
    modelo_gc = keras.Model(modelo.inputs, [modelo.get_layer("top_activation").output, modelo.output])
    x = tf.convert_to_tensor(imagen[None].astype("float32"))
    with tf.GradientTape() as cinta:
        activ, prob = modelo_gc(x, training=False)
        k = int(tf.argmax(prob[0]))
        puntaje = prob[:, k]
    grad = tf.cast(cinta.gradient(puntaje, activ), tf.float32)
    activ = tf.cast(activ, tf.float32)
    pesos = tf.reduce_mean(grad, axis=(1, 2))
    mapa = tf.nn.relu(tf.reduce_sum(activ * pesos[:, None, None, :], axis=-1))[0]
    mapa = mapa / (tf.reduce_max(mapa) + 1e-8)
    return tf.image.resize(mapa[..., None], (TAM, TAM)).numpy()[..., 0], k


def casos_gradcam(tabla):
    """Por clase real: el acierto con mayor probabilidad y el error con mayor probabilidad de la clase predicha."""
    prob = tabla[[f"p_{c}" for c in CLASES_CAMPO]].to_numpy()
    filas = []
    for c in CLASES_CAMPO:
        sub = tabla.index[tabla["clase"] == c]
        aciertos = [i for i in sub if tabla.at[i, "prediccion"] == c]
        errores = [i for i in sub if tabla.at[i, "prediccion"] != c]
        if aciertos:
            filas.append(("acierto", c, max(aciertos, key=lambda i: prob[i].max())))
        if errores:
            filas.append(("error", c, max(errores, key=lambda i: prob[i].max())))
    return filas


for e in ("s1", "s2"):
    s = mejor_semilla(e)
    ruta_modelo = MODELOS / f"{e}_s{s}.keras" if s else None
    if not s or not ruta_modelo.exists():
        RESUMEN["pendientes"].append(f"E4 · Grad-CAM de {e.upper()}")
        continue
    keras.backend.clear_session()
    modelo = copia_float32(keras.models.load_model(ruta_modelo))
    tabla = leer_corrida(f"{e}_s{s}")["prueba"]          # fila i = foto IDX_DEP["prueba"][i]
    casos = casos_gradcam(tabla)
    fig, ejes = plt.subplots(len(CLASES_CAMPO), 4, figsize=(10, 2.6 * len(CLASES_CAMPO)))
    for ax in ejes.ravel():
        ax.axis("off")
    registro = []
    for tipo, c, i in casos:
        fila = CLASES_CAMPO.index(c); col = 0 if tipo == "acierto" else 2
        img = X_campo[IDX_DEP["prueba"][i]]
        mapa, k = gradcam(modelo, img)
        ejes[fila, col].imshow(img)
        ejes[fila, col].set_title(f"{NOMBRE[c]} · {tipo}", fontsize=8)
        ejes[fila, col + 1].imshow(img); ejes[fila, col + 1].imshow(mapa, cmap="jet", alpha=0.45)
        ejes[fila, col + 1].set_title(f"predicha: {NOMBRE[CLASES_CAMPO[k]]} "
                                      f"({tabla.loc[i, f'p_{CLASES_CAMPO[k]}']:.2f})", fontsize=8)
        registro.append({"estrategia": e, "semilla": s, "tipo": tipo, "clase_real": c,
                         "prediccion": CLASES_CAMPO[k], "archivo": tabla.at[i, "archivo"],
                         "probabilidad": round(float(tabla.loc[i, f"p_{CLASES_CAMPO[k]}"]), 4)})
    plt.suptitle(f"Grad-CAM · {NOMBRE_ESTRATEGIA[e]} (semilla {s})", y=1.0)
    plt.tight_layout(); plt.savefig(FIGURAS / f"e4_gradcam_{e}.png", dpi=130, bbox_inches="tight"); plt.show()
    pd.DataFrame(registro).to_csv(RESULTADOS / f"e4_gradcam_{e}.csv", index=False)
    del modelo

# Confusiones más frecuentes de cada estrategia (pares real → predicha, suma de las semillas).
confusiones = []
for e, mc in mc_e2.items():
    for i in range(K):
        for j in range(K):
            if i != j and mc[i, j]:
                confusiones.append({"estrategia": e, "real": NOMBRE[CLASES_CAMPO[i]],
                                    "predicha": NOMBRE[CLASES_CAMPO[j]], "casos": int(mc[i, j]),
                                    "fraccion_de_la_clase": round(mc[i, j] / mc[i].sum(), 4)})
if confusiones:
    conf = pd.DataFrame(confusiones).sort_values(["estrategia", "casos"], ascending=[True, False])
    conf.to_csv(RESULTADOS / "e4_confusiones.csv", index=False)
    display(conf.groupby("estrategia").head(5))
    RESUMEN["e4_confusiones_principales"] = conf.groupby("estrategia").head(5).to_dict(orient="records")

## 11. Curvas de entrenamiento

In [ ]:
for e in ESTRATEGIAS:
    rs = corridas_de(e)
    if not rs:
        continue
    fig, ejes = plt.subplots(1, 2, figsize=(12, 3.4))
    for s in rs:
        h = pd.read_csv(CORRIDAS_DIR / f"{e}_s{s}" / "historial.csv")
        x = np.arange(1, len(h) + 1)
        ejes[0].plot(x, h["loss"], lw=1, label=f"entrenamiento s{s}")
        ejes[0].plot(x, h["val_loss"], lw=1.6, ls="--", label=f"validación s{s}")
        ejes[1].plot(x, h["accuracy"], lw=1); ejes[1].plot(x, h["val_accuracy"], lw=1.6, ls="--")
        if (h["fase"] == "cabeza").any():
            for ax in ejes:
                ax.axvline((h["fase"] == "cabeza").sum() + 0.5, color="gray", lw=0.8)
    ejes[0].set_title(f"{e.upper()} · pérdida"); ejes[1].set_title(f"{e.upper()} · exactitud")
    for ax in ejes:
        ax.set_xlabel("época")
    ejes[0].legend(fontsize=7, ncol=2)
    plt.tight_layout(); plt.savefig(FIGURAS / f"curvas_{e}.png", dpi=120); plt.show()

## 12. Resumen de resultados

`resultados/resumen.json` reúne todas las cifras que se reportan en el artículo. El modelo de la DEMO es el de la
estrategia con mayor F1 macro medio de **validación** y, dentro de ella, la semilla con mayor F1 de validación.

In [ ]:
corridas_hechas = [f"{e}_s{s}" for e, s in CORRIDAS if (CORRIDAS_DIR / f"{e}_s{s}" / "meta.json").exists()]
RESUMEN["corridas_hechas"] = corridas_hechas
RESUMEN["corridas_faltantes"] = [f"{e}_s{s}" for e, s in CORRIDAS if f"{e}_s{s}" not in corridas_hechas]
RESUMEN["tiempos_min"] = {n: round(json.loads((CORRIDAS_DIR / n / "meta.json").read_text(encoding="utf-8"))
                                   ["segundos_entrenamiento"] / 60, 1) for n in corridas_hechas}

candidatas = {e: corridas_de(e) for e in ("s0", "s1", "s2")}
candidatas = {e: r for e, r in candidatas.items() if r}
if candidatas:
    e_demo = max(candidatas, key=lambda e: np.mean([r["meta"]["f1_macro_validacion"] for r in candidatas[e].values()]))
    s_demo = mejor_semilla(e_demo)
    if (MODELOS / f"{e_demo}_s{s_demo}.keras").exists():
        shutil.copy2(MODELOS / f"{e_demo}_s{s_demo}.keras", MODELOS / "modelo_demo.keras")
    RESUMEN["modelo_demo"] = {"corrida": f"{e_demo}_s{s_demo}", "criterio": "mayor F1 macro de validación",
                              "f1_macro_validacion": candidatas[e_demo][s_demo]["meta"]["f1_macro_validacion"]}

ENTORNO["fecha_fin"] = datetime.now().isoformat(timespec="seconds")
(RESULTADOS / "entorno.json").write_text(json.dumps(ENTORNO, indent=1, ensure_ascii=False), encoding="utf-8")
(RESULTADOS / "resumen.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False, default=float),
                                         encoding="utf-8")
archivo_zip = shutil.make_archive(str(TRABAJO / "resultados_TF_Grupo3"), "zip", RESULTADOS)
print("Corridas hechas:", len(corridas_hechas), "de", len(CORRIDAS), "· faltan:", RESUMEN["corridas_faltantes"] or "ninguna")
print("Pendientes:", RESUMEN["pendientes"] or "ninguno")
print("Resultados comprimidos en", archivo_zip)

## 13. DEMO · prueba del modelo sin reentrenar

Carga el modelo guardado (`modelos/modelo_demo.keras` en Drive o, si no está, la versión publicada en el
repositorio) y clasifica cinco fotografías de `demo/`. Las cuatro primeras son de la partición de **prueba**
de campo (ningún modelo las vio al entrenar); la cuarta es un caso límite (nematodos, la clase más rara) y la
quinta es un caso atípico: una foto de laboratorio de PlantVillage, con fondo uniforme, que el modelo de campo
no conoce. La función `predecir` además controla los errores de entrada (archivo inexistente o que no es una
imagen) con un mensaje claro en lugar de detener el cuaderno.

In [ ]:
URL_MODELO_DEMO = ("https://github.com/StevenSntg/papa-transferencia-secuencial/releases/download/"
                   "modelo-v1/modelo_demo.keras")


def cargar_modelo_demo():
    ruta = MODELOS / "modelo_demo.keras"
    if not ruta.exists():
        print("No hay modelo en Drive: se descarga la versión publicada en GitHub…")
        ruta = Path("modelo_demo.keras")
        urllib.request.urlretrieve(URL_MODELO_DEMO, ruta)
    return keras.models.load_model(ruta)


def predecir(modelo, ruta):
    """Devuelve las tres clases más probables o un mensaje de error controlado."""
    ruta = Path(ruta)
    if not ruta.is_file():
        return {"error": f"no existe el archivo {ruta}"}
    try:
        img = leer_224(ruta)
    except Exception as e:
        return {"error": f"no es una imagen válida ({type(e).__name__})"}
    p = modelo.predict(img[None].astype("float32"), verbose=0)[0]
    orden = np.argsort(p)[::-1][:3]
    return {"imagen": img, "top3": [(CLASES_CAMPO[i], float(p[i])) for i in orden]}


keras.backend.clear_session()
modelo_demo = cargar_modelo_demo()
casos = pd.read_csv(RAIZ / "demo" / "casos_demo.csv")
LAB_A_CAMPO = {v: k for k, v in CAMPO_A_LAB.items()}   # la foto de laboratorio se espera como su enfermedad en campo
fig, ejes = plt.subplots(1, len(casos), figsize=(4 * len(casos), 4.6))
filas_demo = []
for ax, (_, caso) in zip(ejes, casos.iterrows()):
    r = predecir(modelo_demo, RAIZ / "demo" / caso["archivo"])
    clase, prob = r["top3"][0]
    esperada = caso["clase_esperada"] if caso["conjunto"] == "campo" else LAB_A_CAMPO[caso["clase_esperada"]]
    acierta = clase == esperada
    ax.imshow(r["imagen"]); ax.axis("off")
    ax.set_title(f"Caso {caso['caso']} · esperada: {NOMBRE[esperada]}\npredicha: {NOMBRE[clase]} ({prob:.2f})"
                 + ("" if prob >= 0.5 else "\nconfianza baja: revisar"), fontsize=9,
                 color="darkgreen" if acierta else "darkred")
    filas_demo.append({"caso": caso["caso"], "descripcion": caso["descripcion"], "esperada": NOMBRE[esperada],
                       "predicha": NOMBRE[clase], "probabilidad": round(prob, 3), "acierta": acierta,
                       "top3": ", ".join(f"{NOMBRE[c]} {q:.2f}" for c, q in r["top3"])})
plt.tight_layout(); plt.savefig(FIGURAS / "demo_casos.png", dpi=130, bbox_inches="tight"); plt.show()
demo = pd.DataFrame(filas_demo)
demo.to_csv(RESULTADOS / "demo_casos.csv", index=False)
display(demo)

print("Error controlado:", predecir(modelo_demo, RAIZ / "demo" / "no_existe.jpg"))
print("Error controlado:", predecir(modelo_demo, RAIZ / "requirements.txt"))

In [ ]:
print("Corrida completa." if not RESUMEN["corridas_faltantes"] else
      f"Corrida incompleta: faltan {RESUMEN['corridas_faltantes']}. Vuelva a «Ejecutar todas» para continuar.")
print("Siguiente paso: Archivo → Descargar → Descargar .ipynb (cuaderno ejecutado) y avisar al grupo; "
      f"los resultados están en {RESULTADOS} y comprimidos en {archivo_zip}.")